In [ ]:
### Pseudobulk coverage tracks
library(Seurat)
library(dplyr)
library(Gviz)
library(rtracklayer)

### Set directories
mainDir <- "/data/ebaird/scRNAseq/SCENTINELsep24/"
repDir <- paste0(mainDir, "pseudobulk/")
figDir <- paste0(repDir, "figs/")
tabDir <- paste0(repDir, "tables/")
refsDir <- paste0(mainDir, "refs/")

dir.create(repDir, recursive = TRUE, showWarnings = FALSE)
dir.create(figDir, recursive = TRUE, showWarnings = FALSE)
dir.create(tabDir, recursive = TRUE, showWarnings = FALSE)

In [ ]:
seurat_obj <- readRDS(paste0(mainDir, "QC_clustering/merged_clusters.rds"))
unique(seurat_obj@meta.data$genotype)

In [ ]:
# Create genotype mapping
barcode_mapping <- data.frame(
  cell_barcode = colnames(seurat_obj),
  genotype = seurat_obj@meta.data$genotype,
  sample = seurat_obj@meta.data$condition
) %>%
  mutate(bam_barcode = gsub("_.*", "", cell_barcode))

# Get valid sample-genotype pairs from data
valid_pairs <- barcode_mapping %>%
  distinct(genotype, sample) %>%
  filter(!is.na(genotype), !is.na(sample))

# Write only valid genotype-sample combinations
for (i in 1:nrow(valid_pairs)) {
  geno <- valid_pairs$genotype[i]
  samp <- valid_pairs$sample[i]
  
  barcodes <- barcode_mapping %>%
    filter(genotype == geno, sample == samp) %>%
    pull(bam_barcode)
  
  if (length(barcodes) > 0) {
    writeLines(barcodes, sprintf(paste0(repDir, "%s_%s.txt"), geno, samp))
    message(sprintf("Writing %d barcodes for %s_%s", length(barcodes), geno, samp))
  } else {
    warning(sprintf("No cells found for VALID pair %s_%s", geno, samp))
  }
}

head(barcode_mapping)

### Now run the bash script 8b-bam_pseudobulk.sh to generate BAM and BigWig files

In [ ]:
unique(seurat_obj@meta.data$condition)

In [ ]:
### Visualize coverage tracks

# Get unique genotypes from seurat_obj
samples <- unique(seurat_obj@meta.data$condition)

# Define region
gene_region <- GRanges(
  seqnames = "2L",
  ranges = IRanges(start = 10000, end = 11000),
  strand = "+"
)

# GTF file
gtf_file <- paste0(refsDir, "genes.gtf")
drosophila_gene_annotation <- import(gtf_file)
# Subset annotation to region of interest
drosophila_gene_annotation <- drosophila_gene_annotation[
  as.character(seqnames(drosophila_gene_annotation)) == as.character(seqnames(gene_region)) &
  start(drosophila_gene_annotation) <= end(gene_region) &
  end(drosophila_gene_annotation) >= start(gene_region)
]

options(ucscChromosomeNames=FALSE)
# Create tracks for each genotype
tracks <- lapply(samples, function(sa) {
  DataTrack(
    range = file.path(repDir, paste0(sa, ".bw")),
    name = sa,
    type = "h",
    col = ifelse(sa == samples[1], "darkgreen", "red"),
    lwd = 2
  )
})

gene_track <- GeneRegionTrack(
  drosophila_gene_annotation,
  chromosome = as.character(seqnames(gene_region)),
  start = start(gene_region),
  end = end(gene_region),
  name = "Genes",
  transcriptAnnotation = "symbol"
)

plotTracks(
  c(tracks, list(gene_track)),
  chromosome = as.character(seqnames(gene_region)),
  from = start(gene_region),
  to = end(gene_region),
  title.width = 1.5,
  background.title = "white",
  groupAnnotation = "group"
)


In [ ]:
# Simple specific cluster groups pseudobulking export as csv

Idents(seurat_obj) <- "seurat_clusters"

seurat_obj$bulk_group <- case_when(
  seurat_obj$seurat_clusters %in% c("2","11","12","9") ~ "MetabolicReprogrammed",
  seurat_obj$seurat_clusters %in% c("6","8","7","14","13") ~ "NeuronalProgenitor",
  seurat_obj$seurat_clusters %in% c("5","0","4","10","3","1") ~ "Neural",
  TRUE ~ NA_character_
)

# Remove NA cells first
seurat_subset <- subset(seurat_obj, subset = !is.na(bulk_group))

pseudobulk <- AggregateExpression(
  seurat_subset,
  group.by = "bulk_group",
  assays = "RNA",
  slot = "counts",
  return.seurat = FALSE
)

pb_matrix <- pseudobulk$RNA

write.csv(pb_matrix,
          file = "pseudobulk_3groups.csv",
          row.names = TRUE)